In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/depression-reddit-cleaned/depression_dataset_reddit_cleaned.csv


In [2]:
df = pd.read_csv('/kaggle/input/depression-reddit-cleaned/depression_dataset_reddit_cleaned.csv')

In [3]:
df.head()

,clean_text,is_depression
0,we understand that most people who reply immed...,1
1,welcome to r depression s check in post a plac...,1
2,anyone else instead of sleeping more when depr...,1
3,i ve kind of stuffed around a lot in my life d...,1
4,sleep is my greatest and most comforting escap...,1


In [4]:
import pandas as pd
from transformers import BertTokenizer, BertModel
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

In [5]:
data = df.copy()
texts = data['clean_text'].tolist()
labels = data['is_depression'].tolist()

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')

def get_bert_embeddings(text):
    tokens = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=128)
    outputs = bert_model(**tokens)
    embeddings = outputs.last_hidden_state.mean(dim=1).squeeze().detach().numpy()
    return embeddings

embeddings = np.array([get_bert_embeddings(text) for text in texts])

X_train, X_test, y_train, y_test = train_test_split(embeddings, labels, test_size=0.2, random_state=42)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy}')
print(classification_report(y_test, y_pred, target_names=['Non-depressed', 'Depressed']))


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Accuracy: 0.9663865546218487
               precision    recall  f1-score   support

Non-depressed       0.95      0.98      0.97       783
    Depressed       0.98      0.95      0.97       764

     accuracy                           0.97      1547
    macro avg       0.97      0.97      0.97      1547
 weighted avg       0.97      0.97      0.97      1547



In [15]:
def predict_custom_input(custom_text):
    # Step 1: Get BERT embeddings for the custom input
    custom_embedding = get_bert_embeddings(custom_text).reshape(1, -1)  # Reshape to 2D array
    
    # Step 2: Predict using the trained classifier
    predicted_label = clf.predict(custom_embedding)
    
    # Step 3: Map the predicted label to a human-readable format
    label_map = {0: 'Non-depressed', 1: 'Depressed'}
    predicted_class = label_map[predicted_label[0]]
    
    return predicted_class

# Example usage:
custom_text = "I am very depressed"

prediction = predict_custom_input(custom_text)
print(f"Predicted label for custom input: {prediction}")


Predicted label for custom input: Depressed


In [16]:
from sklearn.svm import SVC

# Initialize the SVM classifier
clf = SVC(kernel='linear')  # You can try 'rbf' or other kernels as well
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy}')
print(classification_report(y_test, y_pred, target_names=['Non-depressed', 'Depressed']))


Accuracy: 0.962508080155139
               precision    recall  f1-score   support

Non-depressed       0.95      0.97      0.96       783
    Depressed       0.97      0.95      0.96       764

     accuracy                           0.96      1547
    macro avg       0.96      0.96      0.96      1547
 weighted avg       0.96      0.96      0.96      1547



In [17]:
from sklearn.ensemble import RandomForestClassifier

# Initialize the Random Forest classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy}')
print(classification_report(y_test, y_pred, target_names=['Non-depressed', 'Depressed']))


Accuracy: 0.9308338720103426
               precision    recall  f1-score   support

Non-depressed       0.89      0.99      0.94       783
    Depressed       0.99      0.87      0.93       764

     accuracy                           0.93      1547
    macro avg       0.94      0.93      0.93      1547
 weighted avg       0.94      0.93      0.93      1547



In [18]:
from sklearn.ensemble import GradientBoostingClassifier

# Initialize the Gradient Boosting classifier
clf = GradientBoostingClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy}')
print(classification_report(y_test, y_pred, target_names=['Non-depressed', 'Depressed']))


Accuracy: 0.9437621202327084
               precision    recall  f1-score   support

Non-depressed       0.92      0.98      0.95       783
    Depressed       0.97      0.91      0.94       764

     accuracy                           0.94      1547
    macro avg       0.95      0.94      0.94      1547
 weighted avg       0.95      0.94      0.94      1547



In [19]:
from sklearn.neighbors import KNeighborsClassifier

# Initialize the KNN classifier
clf = KNeighborsClassifier(n_neighbors=5)  # You can adjust the number of neighbors
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy}')
print(classification_report(y_test, y_pred, target_names=['Non-depressed', 'Depressed']))


Accuracy: 0.8681318681318682
               precision    recall  f1-score   support

Non-depressed       0.96      0.78      0.86       783
    Depressed       0.81      0.96      0.88       764

     accuracy                           0.87      1547
    macro avg       0.88      0.87      0.87      1547
 weighted avg       0.88      0.87      0.87      1547



In [1]:
!pip install pandas scikit-learn torch transformers sentence-transformers


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 255.2/255.2 kB 5.1 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 23.0.1 -> 24.2
[notice] To update, run: pip install --upgrade pip


In [4]:
data = pd.read_csv('/kaggle/input/depression-reddit-cleaned/depression_dataset_reddit_cleaned.csv') 
texts = data['clean_text'].tolist()
labels = data['is_depression'].tolist()

In [8]:
!pip install tqdm



[notice] A new release of pip is available: 23.0.1 -> 24.2
[notice] To update, run: pip install --upgrade pip


**DistilBert**

In [21]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from transformers import DistilBertTokenizer, DistilBertModel

# Load the dataset
data = pd.read_csv('/kaggle/input/depression-reddit-cleaned/depression_dataset_reddit_cleaned.csv') 
texts = data['clean_text'].tolist()
labels = data['is_depression'].tolist()

# Initialize DistilBERT
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
distilbert_model = DistilBertModel.from_pretrained('distilbert-base-uncased')

def get_distilbert_embeddings(text):
    tokens = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=128)
    outputs = distilbert_model(**tokens)
    embeddings = outputs.last_hidden_state.mean(dim=1).squeeze().detach().numpy()
    return embeddings

# Use tqdm for progress visualization
embeddings = np.array([get_distilbert_embeddings(text) for text in tqdm(texts)])

# Split the data
X_train, X_test, y_train, y_test = train_test_split(embeddings, labels, test_size=0.2, random_state=42)

# Compute class weights
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(enumerate(class_weights))

# Logistic Regression with class weights
clf = LogisticRegression(max_iter=1000, class_weight=class_weights_dict)
clf.fit(X_train, y_train)

# Predictions
y_pred = clf.predict(X_test)

# Evaluation
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy}')
print(classification_report(y_test, y_pred, target_names=['Non-depressed', 'Depressed']))


100%|██████████| 7731/7731 [02:43<00:00, 47.25it/s]


Accuracy: 0.9657401422107305
               precision    recall  f1-score   support

Non-depressed       0.96      0.98      0.97       783
    Depressed       0.98      0.95      0.96       764

     accuracy                           0.97      1547
    macro avg       0.97      0.97      0.97      1547
 weighted avg       0.97      0.97      0.97      1547



**Roberta**

In [28]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from transformers import RobertaTokenizer, RobertaModel

# Load the dataset
data = pd.read_csv('/kaggle/input/depression-reddit-cleaned/depression_dataset_reddit_cleaned.csv') 
texts = data['clean_text'].tolist()
labels = data['is_depression'].tolist()

# Initialize RoBERTa
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
roberta_model = RobertaModel.from_pretrained('roberta-base')

def get_roberta_embeddings(text):
    tokens = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=128)
    outputs = roberta_model(**tokens)
    embeddings = outputs.last_hidden_state.mean(dim=1).squeeze().detach().numpy()
    return embeddings

# Use tqdm for progress visualization
embeddings = np.array([get_roberta_embeddings(text) for text in tqdm(texts)])

# Split the data
X_train, X_test, y_train, y_test = train_test_split(embeddings, labels, test_size=0.2, random_state=42)

# Compute class weights
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(enumerate(class_weights))

# Logistic Regression with class weights
clf = LogisticRegression(max_iter=1000, class_weight=class_weights_dict)
clf.fit(X_train, y_train)

# Predictions
y_pred = clf.predict(X_test)

# Evaluation
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy}')
print(classification_report(y_test, y_pred, target_names=['Non-depressed', 'Depressed']))


/usr/local/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 7731/7731 [05:00<00:00, 25.71it/s]


Accuracy: 0.9644473173884939
               precision    recall  f1-score   support

Non-depressed       0.95      0.98      0.97       783
    Depressed       0.98      0.95      0.96       764

     accuracy                           0.96      1547
    macro avg       0.96      0.96      0.96      1547
 weighted avg       0.96      0.96      0.96      1547



**Electra**

In [30]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from transformers import ElectraTokenizer, ElectraModel

# Load the dataset
data = pd.read_csv('/kaggle/input/depression-reddit-cleaned/depression_dataset_reddit_cleaned.csv') 
texts = data['clean_text'].tolist()
labels = data['is_depression'].tolist()

# Initialize ELECTRA
tokenizer = ElectraTokenizer.from_pretrained('google/electra-base-discriminator')
electra_model = ElectraModel.from_pretrained('google/electra-base-discriminator')

def get_electra_embeddings(text):
    tokens = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=128)
    outputs = electra_model(**tokens)
    embeddings = outputs.last_hidden_state.mean(dim=1).squeeze().detach().numpy()
    return embeddings

# Use tqdm for progress visualization
embeddings = np.array([get_electra_embeddings(text) for text in tqdm(texts)])

# Split the data
X_train, X_test, y_train, y_test = train_test_split(embeddings, labels, test_size=0.2, random_state=42)

# Compute class weights
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(enumerate(class_weights))

# Logistic Regression with class weights
clf = LogisticRegression(max_iter=1000, class_weight=class_weights_dict)
clf.fit(X_train, y_train)

# Predictions
y_pred = clf.predict(X_test)

# Evaluation
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy}')
print(classification_report(y_test, y_pred, target_names=['Non-depressed', 'Depressed']))


100%|██████████| 7731/7731 [05:19<00:00, 24.19it/s]


Accuracy: 0.9379444085326438
               precision    recall  f1-score   support

Non-depressed       0.93      0.95      0.94       783
    Depressed       0.94      0.93      0.94       764

     accuracy                           0.94      1547
    macro avg       0.94      0.94      0.94      1547
 weighted avg       0.94      0.94      0.94      1547



In [3]:
# Install sentence-transformers if not already installed
!pip install sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 255.2/255.2 kB 7.9 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.0.1 -> 24.2
[notice] To update, run: pip install --upgrade pip


**Sentence Transformers**

In [4]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from sentence_transformers import SentenceTransformer

# Load the dataset
data = pd.read_csv('/kaggle/input/depression-reddit-cleaned/depression_dataset_reddit_cleaned.csv') 
texts = data['clean_text'].tolist()
labels = data['is_depression'].tolist()

# Initialize the Sentence Transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Use tqdm for progress visualization when generating embeddings
embeddings = model.encode(texts, show_progress_bar=True)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(embeddings, labels, test_size=0.2, random_state=42)

# Compute class weights
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(enumerate(class_weights))

# Logistic Regression with class weights
clf = LogisticRegression(max_iter=1000, class_weight=class_weights_dict)
clf.fit(X_train, y_train)

# Predictions
y_pred = clf.predict(X_test)

# Evaluation
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy}')
print(classification_report(y_test, y_pred, target_names=['Non-depressed', 'Depressed']))


/usr/local/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.10/site-packages/torch_xla/__init__.py:202: UserWarning: `tensorflow` can conflict with `torch-xla`. Prefer `tensorflow-cpu` when using PyTorch/XLA. To silence this warning, `pip uninstall -y tensorflow && pip install tensorflow-cpu`. If you are in a notebook environment such as Colab or Kaggle, restart your notebook runtime afterwards.
  warnings.warn(
E0000 00:00:1729481823.759631      13 common_lib.cc:798] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:479
E1021 03:37:03.806889532     300 oauth2_credentials.

Accuracy: 0.9618616677440207
               precision    recall  f1-score   support

Non-depressed       0.95      0.98      0.96       783
    Depressed       0.98      0.94      0.96       764

     accuracy                           0.96      1547
    macro avg       0.96      0.96      0.96      1547
 weighted avg       0.96      0.96      0.96      1547

